# Saltwater Bubble Problem

This notebook simulates the sinking of a circular blob ("bubble") of salty water inside a domain of fresh water. It demonstrates how MODFLOW 6 couples **variable-density groundwater flow** with **solute transport** to reproduce buoyancy-driven flow.

## What this notebook covers

You will build a two-dimensional cross-section model, place a disk of dense saltwater high in an otherwise fresh domain, run the coupled flow-and-transport simulation, and visualize how the dense blob sinks. By the end you will be able to build a coupled groundwater flow (**GWF**) and groundwater transport (**GWT**) model, switch on density-driven flow with the Buoyancy (**BUY**) package, and animate the results.

**The key idea: buoyancy-driven flow.** In an ordinary groundwater model the water has a single, constant density, so it moves only where there is a hydraulic gradient. Here the dissolved salt makes water heavier: the more salt, the denser the water. The Buoyancy (BUY) package adds this density effect to the flow equation, so the salty blob sinks under its own weight and drags fresh water up around it - motion that a constant-density model could never produce. The salt is carried along by that motion, which changes the density again, so flow and transport must be solved together.

This example will touch on a few topics:
- How does grid refinement affect the simulation?
- How does adaptive time stepping affect the simulation?
- How does the advection scheme affect the simulation?
- How does dispersion affect the simulation?

## Import packages

Import the packages used throughout the notebook.

In [ ]:
import pathlib as pl

import flopy
import matplotlib
import matplotlib.animation
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

In [ ]:
ws = pl.Path("./models/bubble")

## Define a helper that builds the grid

Wrap grid construction in a helper function so you can change the domain size and resolution in one place. `make_grid()` returns the cell spacings (`delr`, `delc`), the `top`, and the layer-bottom elevations (`botm`) for the requested number of layers, rows, and columns.

In [ ]:
def make_grid(nlay, nrow, ncol, lx, top0):
    top = np.zeros((nrow, ncol), dtype=float)
    botm = np.zeros((nlay, nrow, ncol), dtype=float)

    delr = lx / ncol
    delc = 1.0
    delv = top0 / nlay
    top[:, :] = top0

    for k in range(nlay):
        tp = top0
        if k > 0:
            tp = botm[k - 1, 0, 0]
        botm[k, :, :] = tp - delv

    delr = np.ones(ncol) * delr
    delc = np.ones(nrow) * delc
    return (delr, delc, top, botm)

#### Define a function that builds the simulation

You will run this problem several times with different settings, so wrap the whole model build in a function. Passing in the grid size, transport options, and time stepping keeps each run to a single call.

The function assembles a complete coupled simulation: a groundwater flow (GWF) model, a groundwater transport (GWT) model, one iterative solver for each, and a **GWF-GWT exchange** that links them so heads drive transport and concentrations feed back into flow. Density coupling is switched on with the Buoyancy (BUY) package, whose `packagedata` is one tuple per dissolved species:

```python
# (species_index, drhodc, reference_conc, transport_model_name, aux_species_name)
(0, 0.7, 0.0, gwtname, "concentration")
```

Here `drhodc = 0.7` sets how much the water density rises per unit of concentration, and the tuple ties that density to the concentration simulated by the GWT model. Adaptive time stepping (**ATS**) is added when `ats_percel` is provided, letting MODFLOW shrink or grow the time step to keep the solution stable.

In [ ]:
def create_simulation(
    ws,
    name,
    nlay,
    nrow,
    ncol,
    lx,
    top0,
    time_end,
    hydraulic_conductivity,
    specific_storage,
    porosity,
    alpha_l,
    alpha_t,
    cfresh,
    csalt,
    nstp=100,
    hstrt=None,
    cstrt=None,
    converge_continue=False,
    ats_percel=None,
    adv_scheme="UPSTREAM",
    verbosity_level=1,
):
    delr, delc, top, botm = make_grid(nlay, nrow, ncol, lx, top0)

    nper = 1
    perlen = nper * [time_end]
    tdis_rc = []
    for i in range(nper):
        tdis_rc.append((perlen[i], nstp, 1.0))

    nouter, ninner = 100, 300
    hclose, rclose, relax = 1e-7, 1e-5, 0.97

    # build MODFLOW 6 files
    sim = flopy.mf6.MFSimulation(
        sim_name=name,
        version="mf6",
        verbosity_level=verbosity_level,
        sim_ws=ws,
    )
    if converge_continue:
        sim.name_file.continue_ = True
    # create tdis package
    tdis = flopy.mf6.ModflowTdis(sim, time_units="DAYS", nper=nper, perioddata=tdis_rc)

    if ats_percel is not None:
        # set dt0, dtmin, dtmax, dtadj, dtfailadj
        dt0 = 0.01
        dtmin = 1.0e-5
        dtmax = perlen
        dtadj = 2.0
        dtfailadj = 5.0
        ats_filerecord = name + ".ats"
        atsperiod = [(0, dt0, dtmin, dtmax[i], dtadj, dtfailadj) for i in range(nper)]
        tdis.ats.initialize(
            maxats=len(atsperiod),
            perioddata=atsperiod,
            filename=ats_filerecord,
        )

    # create gwf model
    gwfname = "gwf_" + name
    gwtname = "gwt_" + name

    gwf = flopy.mf6.ModflowGwf(sim, modelname=gwfname, newtonoptions=True)
    imsgwf = flopy.mf6.ModflowIms(
        sim,
        ats_outer_maximum_fraction=0.0,
        print_option="ALL",
        outer_hclose=hclose,
        outer_maximum=nouter,
        under_relaxation="DBD",
        inner_maximum=ninner,
        inner_hclose=hclose,
        rcloserecord=rclose,
        linear_acceleration="BICGSTAB",
        scaling_method="NONE",
        reordering_method="NONE",
        relaxation_factor=relax,
        no_ptcrecord=True,
        filename=f"{gwfname}.ims",
    )
    single_matrix = False
    if single_matrix:
        sim.register_ims_package(imsgwf, [gwfname, gwtname])
    else:
        sim.register_ims_package(imsgwf, [gwfname])

    _ = flopy.mf6.ModflowGwfdis(
        gwf, nlay=nlay, nrow=nrow, ncol=ncol, delr=delr, delc=delc, top=top, botm=botm
    )

    # initial conditions
    if hstrt is None:
        hstrt = top0
    else:
        hstrt = {
            "filename": "hstrt.bin",
            "factor": 1.0,
            "data": hstrt,
            "binary": "True",
        }
    _ = flopy.mf6.ModflowGwfic(gwf, strt=hstrt)

    # node property flow
    _ = flopy.mf6.ModflowGwfnpf(
        gwf,
        xt3doptions=False,
        save_flows=True,
        save_specific_discharge=True,
        icelltype=0,
        k=hydraulic_conductivity,
    )

    _ = flopy.mf6.ModflowGwfsto(gwf, iconvert=1, sy=porosity, ss=specific_storage)

    pd = [(0, 0.7, 0.0, gwtname, "concentration")]
    _ = flopy.mf6.ModflowGwfbuy(gwf, packagedata=pd)

    # output control
    saverecord = {
        0: [("HEAD", "ALL"), ("BUDGET", "ALL")],
        1: [("HEAD", "ALL"), ("BUDGET", "ALL")],
        nper - 1: [("HEAD", "ALL"), ("BUDGET", "ALL")],
    }
    printrecord = {
        0: [("HEAD", "LAST"), ("BUDGET", "LAST")],
        1: None,
        nper - 1: [("HEAD", "ALL"), ("BUDGET", "LAST")],
    }
    _ = flopy.mf6.ModflowGwfoc(
        gwf,
        budgetcsv_filerecord=f"{gwfname}.bud.csv",
        budget_filerecord=f"{gwfname}.cbc",
        head_filerecord=f"{gwfname}.hds",
        headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
        saverecord=saverecord,
        printrecord=printrecord,
    )

    # create gwt model
    gwt = flopy.mf6.MFModel(
        sim,
        model_type="gwt6",
        modelname=gwtname,
        model_nam_file=f"{gwtname}.name",
    )

    if not single_matrix:
        imsgwt = flopy.mf6.ModflowIms(
            sim,
            ats_outer_maximum_fraction=0.0,
            print_option="ALL",
            outer_hclose=hclose,
            outer_maximum=nouter,
            under_relaxation="NONE",
            inner_maximum=ninner,
            inner_hclose=hclose,
            rcloserecord=rclose,
            linear_acceleration="BICGSTAB",
            scaling_method="NONE",
            reordering_method="NONE",
            relaxation_factor=relax,
            filename=f"{gwtname}.ims",
        )
        sim.register_ims_package(imsgwt, [gwt.name])

    _ = flopy.mf6.ModflowGwtdis(
        gwt, nlay=nlay, nrow=nrow, ncol=ncol, delr=delr, delc=delc, top=top, botm=botm
    )

    # initial conditions
    if cstrt is None:
        cstrt = csalt
    else:
        cstrt = {
            "filename": "cstrt.bin",
            "factor": 1.0,
            "data": cstrt,
            "binary": "True",
        }
    _ = flopy.mf6.ModflowGwtic(gwt, strt=cstrt)

    # advection
    _ = flopy.mf6.ModflowGwtadv(gwt, ats_percel=ats_percel, scheme=adv_scheme)

    # dispersion
    _ = flopy.mf6.ModflowGwtdsp(
        gwt, xt3d_off=False, diffc=0.0, alh=alpha_l, ath1=alpha_t
    )

    # mass storage and transfer
    porosity = porosity
    _ = flopy.mf6.ModflowGwtmst(gwt, porosity=porosity)

    # output control
    saverecord = {
        0: [("CONCENTRATION", "ALL"), ("BUDGET", "LAST")],
        1: [("CONCENTRATION", "ALL")],
        nper - 1: [("CONCENTRATION", "ALL"), ("BUDGET", "LAST")],
    }
    printrecord = {
        0: [("CONCENTRATION", "LAST"), ("BUDGET", "LAST")],
        1: None,
        nper - 1: [("CONCENTRATION", "ALL"), ("BUDGET", "LAST")],
    }
    _ = flopy.mf6.ModflowGwtoc(
        gwt,
        budget_filerecord=f"{gwtname}.bud",
        budgetcsv_filerecord=f"{gwtname}.bud.csv",
        concentration_filerecord=f"{gwtname}.ucn",
        concentrationprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
        saverecord=saverecord,
        printrecord=printrecord,
    )

    # GWF GWT exchange
    _ = flopy.mf6.ModflowGwfgwt(
        sim,
        exgtype="GWF6-GWT6",
        exgmnamea=gwfname,
        exgmnameb=gwtname,
        filename=f"{name}.gwfgwt",
    )

    return sim

#### Define a function that reads the results

Pull the simulated heads, concentrations, output times, and specific-discharge (flow) data out of the two models with `gwf.output.head()`, `gwt.output.concentration()`, and `gwf.output.budget()`. Returning them together keeps the plotting cells short.

In [ ]:
def get_results(ws, sim):
    modelnames = list(sim.model_names)
    gwfname = modelnames[0]
    gwtname = modelnames[1]
    gwf = sim.get_model(gwfname)
    gwt = sim.get_model(gwtname)

    head = gwf.output.head().get_alldata()
    conc = gwt.output.concentration().get_alldata()
    times = gwt.output.concentration().get_times()

    budobj = gwf.output.budget()
    spdis = budobj.get_data(text="DATA-SPDIS")

    return head, conc, spdis, times

#### Define a function that plots concentration

Draw the salt concentration on a cross section with `flopy.plot.PlotCrossSection()` and `.plot_array()`. Optionally overlay flow streamlines or specific-discharge vectors to see the buoyancy-driven circulation that the sinking bubble sets up.

In [ ]:
def add_streamplot(gwf, spdis, ax):
    x = gwf.modelgrid.xcellcenters
    y = gwf.modelgrid.zcellcenters
    X, Y = np.meshgrid(x, y[:, 0, 0])
    u = spdis["qx"]
    u = u.reshape(X.shape)
    v = spdis["qz"]
    v = v.reshape(X.shape)

    Y = Y[::-1, :]
    u = u[::-1, :]
    v = v[::-1, :]

    _ = ax.streamplot(X, Y, u, v, color="yellow", linewidth=0.1, arrowsize=0.35)


def make_figure(
    gwf,
    head,
    conc,
    spdis,
    t,
    top0,
    csalt=35.0,
    ax2dict=None,
    asoln=None,
    streamplot=False,
    vectors=False,
    figname=None,
):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal")

    pxs = flopy.plot.PlotCrossSection(model=gwf, line={"row": 0}, ax=ax)
    pxs.plot_grid(alpha=1.0, lw=0.1)
    cb = pxs.plot_array(conc, vmin=0.0, vmax=csalt, cmap="jet")

    cbar = plt.colorbar(cb, shrink=0.5)
    cbar.ax.get_yaxis().labelpad = 15
    cbar.ax.set_ylabel("CONCENTRATION, IN GRAMS PER LITER", rotation=90)

    if vectors:
        pxs.plot_specific_discharge(
            spdis,
            head=head,
            color="white",
            kstep=1,
            hstep=1,
            pivot="mid",
            minlength=0.5,
            scale=0.1,
            width=0.001,
            headwidth=3.0,
            headlength=5.0,
            headaxislength=4.5,
            minshaft=0.01,
            zorder=10,
            alpha=0.50,
        )

    if streamplot:
        add_streamplot(gwf, spdis, ax)

    ax.set_title(f"TIME = {t / 365.0:.2f} years")
    ax.set_xlabel("DISTANCE, IN METERS")
    ax.set_ylabel("ELEVATION, IN METERS")

    plt.tight_layout()

    if figname is not None:
        plt.savefig(figname, dpi=300)
        plt.close(fig)

    return ax

#### Clear any old variables

Reset the result variables to `None` so that a later cell cannot accidentally plot data left over from an earlier run.

In [ ]:
# Protection
sim = None
sim0 = None
gwf = None
head = None
head0 = None
conc = None
conc0 = None

# Set up and run the simulation

Set the discretization and model parameters for the base run. The domain is 100 by 100 meters resolved on a 20-layer by 20-column cross section (`nrow = 1`), with fresh water at concentration 0 and saltwater at 35 g/L. This first run uses pure advection with the upstream scheme and no dispersion (`alpha_l = alpha_t = 0`); you will relax those choices in the exercises.

In [ ]:
nrow = 1
nlay = 20
ncol = 20
lx = 100.0
lz = 100.0
nstp = 100
time_end = 2000.0
ats_percel = None
cfresh = 0.0
csalt = 35.0
porosity = 0.20
hydraulic_conductivity = 1.0  # m/day
specific_storage = 0.008  # per meter
alpha_l = 0.0  # meter
alpha_t = 0.0  # meter
adv_scheme = "UPSTREAM"

#### Set the initial saltwater bubble

Build the starting concentration array: a circular patch of saltwater (concentration 35) high in the domain, surrounded by fresh water (0). Use a temporary `StructuredGrid` and `GridIntersect` to find which cells fall inside a circle of radius 10 centered near the top, then mark those cells salty. The `plt.imshow()` preview shows the bubble before the simulation starts.

In [ ]:
import flopy.discretization as fgrid
from flopy.utils.gridintersect import GridIntersect
from shapely.geometry import Polygon

dx = np.empty(ncol)
dx.fill(lx / ncol)
dy = np.empty(nlay)
dy.fill(lz / nlay)

# use a temporary grid for finding cells within saltwater circle
sgr = fgrid.StructuredGrid(dx, dy, top=None, botm=None)

radius = 10.0
xmid = lx / 2.0
ymid = 90.0

theta = np.arange(0.0, 2 * np.pi, 0.1)
x = radius * np.cos(theta) + xmid
y = radius * np.sin(theta) + ymid
p = Polygon(shell=[(x, y) for x, y in zip(x, y, strict=False)])
ix = GridIntersect(sgr)
result = ix.intersect(p, shapetype="polygon")
cstrt = np.zeros((nlay, nrow, ncol))
for k, j in result["cellids"]:
    cstrt[k, 0, j] = csalt
plt.imshow(cstrt[:, 0, :])

#### Create and write the base simulation

Call the builder function with the base-case parameters to assemble the coupled simulation, then write the MODFLOW 6 input files with `sim0.write_simulation()`.

In [ ]:
name = "bubble"
sim0 = create_simulation(
    ws,
    name,
    nlay,
    nrow,
    ncol,
    lx,
    lz,
    time_end,
    hydraulic_conductivity,
    specific_storage,
    porosity,
    alpha_l,
    alpha_t,
    cfresh,
    csalt,
    nstp=nstp,
    cstrt=cstrt,
    ats_percel=ats_percel,
    adv_scheme=adv_scheme,
    verbosity_level=1,
)
sim0.write_simulation()

#### Run the simulation

Run MODFLOW 6 with `sim0.run_simulation()`. The `assert` stops the notebook if the model does not converge, so wait for the "Simulation ran successfully!" message before continuing to the next block.

In [ ]:
print("Starting simulation")
print("Please wait for the simulation to complete.")
success, buff = sim0.run_simulation(silent=True, report=False)
assert success, "model failed"

print("\nSimulation ran successfully!")

#### Load results and plot the final concentration

Read the results with the helper function, grab the flow (`gwf`) and transport (`gwt`) models, and plot the concentration at the last time step (`kper = -1`) with flow streamlines overlaid.

In [ ]:
# load results
head0, conc0, spdis0, times0 = get_results(ws, sim0)
gwf = sim0.get_model("gwf_bubble")
gwt = sim0.get_model("gwt_bubble")

In [ ]:
kper = -1
ax = make_figure(
    gwf, head0[kper], conc0[kper], spdis0[kper], times0[kper], lz, streamplot=True
)

# Create an animation

Replay the whole simulation as an animation. Load the concentration at every saved time step, then use `matplotlib.animation.FuncAnimation` to step through them. The printout also reports how many time steps the run took - a useful number to watch when you switch on adaptive time stepping in the exercises.

In [ ]:
# load output
qx, qy, qz = None, None, None
head_all = gwf.output.head().get_alldata()
bud = gwf.output.budget()
times = bud.times
spdis = bud.get_data(text="DATA-SPDIS")[0]
qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf)
conc_all = gwt.output.concentration().get_alldata()
figsize = (8, 8)

In [ ]:
times_to_show = times[::10]
ntimes = len(times_to_show)
bud = gwf.output.budget()
print(f"Number of time steps: {len(times)}")
print(f"{ntimes} frames will be added to animation.")

**What to look for.** Watch the bubble fall through time. Early on it is a compact disk; as the simulation runs, buoyancy pulls it downward and it spreads into a sinking plume. Because the base case has no dispersion, the sharp concentration front is smeared only by numerical mixing from the upstream advection scheme - something you can change in the exercises.

In [ ]:
def get_title(time_in_days):
    time_in_years = time_in_days / 365.0
    return f"Time = {time_in_years:0.2f} years"


nplotrows = 1
nplotcols = 1
fig, axes = plt.subplots(
    nrows=nplotrows, ncols=nplotcols, figsize=figsize, layout="constrained"
)
ax = axes
ax.set_ylabel(r"z")
title = ax.set_title(get_title(times[0]))
ax.set_xlabel(r"x")
# ax.set_xlim(0, 2.0)
# ax.set_ylim(0, 1.0)
ax.set_aspect(1.0)

# # plot persistent items
plot_array_dict = {
    "cmap": "jet",
    "masked_values": [1.0e30],
}
colorbar_text_size = 10
pmv = flopy.plot.PlotCrossSection(gwt, line={"row": 0}, ax=ax)
pmv.plot_inactive()
pa = pmv.plot_array(conc_all[0], vmin=0.0, vmax=35.0, **plot_array_dict)
cb = plt.colorbar(pa, shrink=0.5)
cb.ax.set_ylabel(
    "salinity (g/L)", rotation=270, fontsize=colorbar_text_size, labelpad=15
)
cb.ax.tick_params(labelsize=colorbar_text_size)


def animate(i):
    print(".", end="", flush=True)
    itime = times.index(times_to_show[i])
    _ = ax.set_title(get_title(times[itime]))
    _ = pmv.plot_array(conc_all[itime], vmin=0.0, vmax=35.0, **plot_array_dict)
    # spdis = bud.get_data(text='DATA-SPDIS')[itime]
    # qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf)
    # pmv.plot_vector(qx, qy, qz, normalize=False, color="white")

    return


ani = matplotlib.animation.FuncAnimation(fig, animate, frames=ntimes)
plt.close()

# Create and show the animation in the notebook
HTML(ani.to_jshtml())

## Recap

In this notebook you:
- Built a coupled groundwater flow (GWF) and transport (GWT) simulation linked by a GWF-GWT exchange.
- Switched on density-driven flow with the Buoyancy (BUY) package so that salty water sinks under its own weight.
- Set a circular saltwater bubble as the initial condition and ran the model to watch it sink.
- Loaded the results and built both a static cross section and an animation of the sinking plume.

Revisit the questions at the top of the notebook — grid resolution, adaptive time stepping, the advection scheme, and dispersion — and rerun `create_simulation()` with different settings to see how each one changes the result.

In [ ]:
# Activate to save as gif
# writer = matplotlib.animation.PillowWriter(fps=20) # Adjust fps as needed
# ani.save('bubble_animation.gif', writer=writer)

In [ ]:
# Plans for exercise:
# Run with specified time steps, upstream advection, no dispersion
# Increase grid resolution (suggest not going over nlay=100, ncol=100)
# Turn on ATS
# Test advection schemes
# Add dispersion